In [2]:
"""
=============================================================================
IoT-RPL 2021 — ROLLING WINDOW FEATURE ENGINEERING
=============================================================================
Reads the 10 raw CSV files directly and engineers behavioral features
using a sliding window over each source node's packet history.

Why this is needed:
  Raw packet-level features (control_type, DOAGID.1, etc.) have ~49%
  accuracy because a single packet from a Blackhole node looks identical
  to a single packet from a Normal node. Attack signatures only emerge
  from patterns over time. Rolling window features capture what a node
  has been DOING over its last N packets — which generalises to new
  networks without memorising node identity.

How it works:
  For each file (processed independently to prevent train/test leakage):
    For each source node ('from' column):
      Sort packets in original order
      For each packet, look at the last WINDOW_SIZE packets from that node
      Compute summary statistics over that window
  The 'from' column is dropped AFTER feature engineering — the model
  never sees node identity, only behavioural patterns.

Engineered features (13):
  roll_total       — total packets sent by this node in window
  roll_n_DIS       — count of DIS control messages in window
  roll_n_DIO       — count of DIO control messages in window
  roll_n_DAO       — count of DAO control messages in window
  roll_ratio_DIS   — DIS / total in window
  roll_ratio_DIO   — DIO / total in window
  roll_ratio_DAO   — DAO / total in window
  roll_unique_to   — unique destination nodes contacted in window
  roll_avg_doagid2 — rolling mean of DOAGID.2 (routing graph activity)
  roll_avg_doagid3 — rolling mean of DOAGID.3
  roll_avg_dio     — rolling mean of DIO_info field
  roll_avg_obj     — rolling mean of object_cont_pt field
  roll_avg_life    — rolling mean of preferred_liftime field

Outputs:
  iot_rpl_engineered_train.csv  — files 0-7, engineered features
  iot_rpl_engineered_test.csv   — files 8-9, engineered features

Leakage safeguards:
  - Rolling windows computed per file — never cross file boundaries
  - 'from' column dropped before saving
  - Test files processed independently from train files
  - No scaler fitted here — scaling happens in ML pipeline as before
=============================================================================
"""

import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

IOT_RPL_DIR   = "IoT-RPL Dataset 2021/"
IOT_RPL_FILES = 10
LABEL_COL     = "label"
WINDOW_SIZE   = 100    # packets to look back per node — change to 50 or 200 to experiment

TRAIN_FILES   = list(range(8))   # files 0-7
TEST_FILES    = [8, 9]           # files 8-9

COL_NAMES = [
    'from', 'to', 'frame_proto', 'protocol', 'control_type', 'type_cont_messg',
    'DOAGID', 'DOAGID.1', 'DOAGID.2', 'DOAGID.3', 'DOAG_info', 'DIO_info',
    'object_cont_pt', 'lifetime', 'prefix_info', 'valid_lifetime',
    'preferred_liftime', 'reserved', 'desti_prefix', 'desti_prefix.1',
    'desti_prefix.2', 'desti_prefix.3', 'label'
]

# Message type detection — these are the string values in the raw files
# control_type column contains: 'DIS', and numeric codes
# type_cont_messg column contains: 'DIO', 'DAO', and numeric codes
DIS_VALUE = 'DIS'
DIO_VALUE = 'DIO'
DAO_VALUE = 'DAO'

# Numeric columns to include in rolling means
NUMERIC_ROLL_COLS = [
    'DOAGID.2', 'DOAGID.3', 'DIO_info', 'object_cont_pt', 'preferred_liftime'
]

ENGINEERED_FEATURE_COLS = [
    'roll_total',
    'roll_n_DIS',
    'roll_n_DIO',
    'roll_n_DAO',
    'roll_ratio_DIS',
    'roll_ratio_DIO',
    'roll_ratio_DAO',
    'roll_unique_to',
    'roll_avg_doagid2',
    'roll_avg_doagid3',
    'roll_avg_dio',
    'roll_avg_obj',
    'roll_avg_life',
]


# ============================================================
# LOAD ONE RAW FILE
# ============================================================

def load_raw_file(file_idx):
    path = IOT_RPL_DIR + f"{file_idx}.csv"
    if file_idx == 0:
        df = pd.read_csv(path, low_memory=False)
    else:
        df = pd.read_csv(path, header=None, names=COL_NAMES, low_memory=False)

    df = df.dropna(subset=[LABEL_COL])
    df['file_id'] = file_idx
    df = df.reset_index(drop=True)
    return df


# ============================================================
# ENGINEER FEATURES FOR ONE FILE
# ============================================================

def engineer_one_file(df, file_idx, window=WINDOW_SIZE):
    """
    Compute rolling window behavioral features per source node.
    Packets are processed in their original order within the file.
    Returns a DataFrame with engineered features + label only.
    """

    # Binary indicator columns for message types
    df['_is_DIS'] = (df['control_type'].astype(str).str.strip() == DIS_VALUE).astype(np.float32)
    df['_is_DIO'] = (df['type_cont_messg'].astype(str).str.strip() == DIO_VALUE).astype(np.float32)
    df['_is_DAO'] = (df['type_cont_messg'].astype(str).str.strip() == DAO_VALUE).astype(np.float32)

    # Coerce numeric columns
    for col in NUMERIC_ROLL_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).clip(-3.4e38, 3.4e38)

    # Coerce 'to' for unique destination count
    df['_to_num'] = pd.to_numeric(df['to'], errors='coerce').fillna(-1)

    # We'll build the engineered rows here
    result_rows = []

    # Group by source node and process in order
    # Using iterative approach for memory efficiency on large files
    nodes = df['from'].unique()
    n_nodes = len(nodes)

    print(f"    File {file_idx}: {len(df):,} packets, {n_nodes} nodes", flush=True)

    for node in nodes:
        mask    = df['from'] == node
        node_df = df[mask].copy()

        n = len(node_df)
        if n == 0:
            continue

        # Precompute arrays for speed
        is_DIS  = node_df['_is_DIS'].values
        is_DIO  = node_df['_is_DIO'].values
        is_DAO  = node_df['_is_DAO'].values
        to_vals = node_df['_to_num'].values
        labels  = node_df[LABEL_COL].values

        doagid2 = node_df['DOAGID.2'].values
        doagid3 = node_df['DOAGID.3'].values
        dio_inf = node_df['DIO_info'].values
        obj_ct  = node_df['object_cont_pt'].values
        pref_lf = node_df['preferred_liftime'].values

        for i in range(n):
            # Window: packets [max(0, i-window+1) .. i] inclusive
            lo = max(0, i - window + 1)

            w_DIS   = is_DIS[lo:i+1]
            w_DIO   = is_DIO[lo:i+1]
            w_DAO   = is_DAO[lo:i+1]
            w_to    = to_vals[lo:i+1]
            w_total = i - lo + 1

            n_DIS = w_DIS.sum()
            n_DIO = w_DIO.sum()
            n_DAO = w_DAO.sum()

            result_rows.append({
                'roll_total'      : w_total,
                'roll_n_DIS'      : n_DIS,
                'roll_n_DIO'      : n_DIO,
                'roll_n_DAO'      : n_DAO,
                'roll_ratio_DIS'  : n_DIS / w_total,
                'roll_ratio_DIO'  : n_DIO / w_total,
                'roll_ratio_DAO'  : n_DAO / w_total,
                'roll_unique_to'  : len(np.unique(w_to[w_to >= 0])),
                'roll_avg_doagid2': doagid2[lo:i+1].mean(),
                'roll_avg_doagid3': doagid3[lo:i+1].mean(),
                'roll_avg_dio'    : dio_inf[lo:i+1].mean(),
                'roll_avg_obj'    : obj_ct[lo:i+1].mean(),
                'roll_avg_life'   : pref_lf[lo:i+1].mean(),
                LABEL_COL         : labels[i],
            })

    result_df = pd.DataFrame(result_rows)
    return result_df


# ============================================================
# PROCESS A LIST OF FILES AND CONCATENATE
# ============================================================

def process_files(file_indices, split_name):
    print(f"\n{'='*70}")
    print(f"PROCESSING {split_name.upper()} FILES: {file_indices}")
    print(f"{'='*70}")

    chunks = []
    t0 = time.time()

    for file_idx in file_indices:
        print(f"\n  Loading file {file_idx}...")
        df_raw = load_raw_file(file_idx)
        print(f"  Engineering features...")
        df_eng = engineer_one_file(df_raw, file_idx)
        chunks.append(df_eng)
        print(f"  Done: {len(df_eng):,} rows engineered")

        # Free memory
        del df_raw

    combined = pd.concat(chunks, ignore_index=True)
    elapsed  = time.time() - t0

    print(f"\n  {split_name} total rows : {len(combined):,}")
    print(f"  {split_name} columns    : {combined.columns.tolist()}")
    print(f"  {split_name} time       : {elapsed/60:.1f} min")
    print(f"\n  Label distribution:")
    for cls, cnt in combined[LABEL_COL].value_counts().items():
        print(f"    {str(cls).ljust(12)}: {cnt:,}  ({cnt/len(combined)*100:.1f}%)")

    return combined


# ============================================================
# SANITY CHECK — verify engineered features look sensible
# ============================================================

def sanity_check(df, name):
    print(f"\n{'='*70}")
    print(f"SANITY CHECK — {name}")
    print(f"{'='*70}")

    feature_cols = ENGINEERED_FEATURE_COLS

    print(f"\n  Per-class mean values (should differ between attack types):\n")
    print(f"  {'Feature'.ljust(20)}", end="")
    classes = sorted(df[LABEL_COL].unique())
    for cls in classes:
        print(f"  {str(cls).ljust(12)}", end="")
    print()
    print("  " + "-" * (20 + 14 * len(classes)))

    for col in feature_cols:
        print(f"  {col.ljust(20)}", end="")
        for cls in classes:
            mean = df[df[LABEL_COL] == cls][col].mean()
            print(f"  {mean:.3f}".ljust(14), end="")
        print()

    print(f"\n  NaN count per feature:")
    nan_counts = df[feature_cols].isna().sum()
    for col, cnt in nan_counts.items():
        if cnt > 0:
            print(f"    {col}: {cnt:,}")
    if nan_counts.sum() == 0:
        print(f"    None — all features clean")

    # Quick single-feature accuracy check
    print(f"\n  Quick leakage check (depth-4 single-feature accuracy):")
    print(f"  Chance = 0.20 for 5 classes. >0.50 = potential signal.")
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.preprocessing import LabelEncoder
    from sklearn.model_selection import cross_val_score

    le = LabelEncoder()
    y  = le.fit_transform(df[LABEL_COL])

    # Sample for speed
    n_sample = min(50_000, len(df))
    idx      = np.random.choice(len(df), n_sample, replace=False)
    df_s     = df.iloc[idx]
    y_s      = y[idx]

    for col in feature_cols:
        vals = df_s[col].fillna(0).values.reshape(-1, 1).astype(np.float32)
        score = cross_val_score(
            DecisionTreeClassifier(max_depth=4, random_state=42),
            vals, y_s, cv=3, n_jobs=-1
        ).mean()
        flag = "  <- good signal" if score > 0.40 else ""
        print(f"    {col.ljust(22)}: {score:.3f}{flag}")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    total_start = time.time()

    print("=" * 70)
    print("IoT-RPL 2021 — ROLLING WINDOW FEATURE ENGINEERING")
    print(f"Window size : {WINDOW_SIZE} packets per node")
    print(f"Train files : {TRAIN_FILES}")
    print(f"Test files  : {TEST_FILES}")
    print("=" * 70)

    # Process train files
    df_train = process_files(TRAIN_FILES, "TRAIN")

    # Process test files
    df_test = process_files(TEST_FILES, "TEST")

    # Sanity checks
    sanity_check(df_train, "TRAIN SET")
    sanity_check(df_test,  "TEST SET")

    # Save
    print(f"\n{'='*70}")
    print("SAVING OUTPUTS")
    print(f"{'='*70}")

    train_path = "iot_rpl_engineered_train.csv"
    test_path  = "iot_rpl_engineered_test.csv"

    df_train.to_csv(train_path, index=False)
    print(f"  Saved: {train_path}  {df_train.shape}")

    df_test.to_csv(test_path, index=False)
    print(f"  Saved: {test_path}   {df_test.shape}")

    print(f"\n  Engineered features ({len(ENGINEERED_FEATURE_COLS)}):")
    for f in ENGINEERED_FEATURE_COLS:
        print(f"    {f}")

    print(f"\n  Total time: {(time.time()-total_start)/60:.1f} min")
    print(f"\n  Next step:")
    print(f"  Update ML pipeline config:")
    print(f"    TRAIN_PATH   = '{train_path}'")
    print(f"    TEST_PATH    = '{test_path}'")
    print(f"    FEATURE_COLS = {ENGINEERED_FEATURE_COLS}")

IoT-RPL 2021 — ROLLING WINDOW FEATURE ENGINEERING
Window size : 100 packets per node
Train files : [0, 1, 2, 3, 4, 5, 6, 7]
Test files  : [8, 9]

PROCESSING TRAIN FILES: [0, 1, 2, 3, 4, 5, 6, 7]

  Loading file 0...
  Engineering features...
    File 0: 1,048,575 packets, 16 nodes
  Done: 1,048,575 rows engineered

  Loading file 1...
  Engineering features...
    File 1: 1,018,083 packets, 16 nodes
  Done: 1,018,083 rows engineered

  Loading file 2...
  Engineering features...
    File 2: 1,029,330 packets, 16 nodes
  Done: 1,029,330 rows engineered

  Loading file 3...
  Engineering features...
    File 3: 1,033,457 packets, 16 nodes
  Done: 1,033,457 rows engineered

  Loading file 4...
  Engineering features...
    File 4: 1,004,792 packets, 16 nodes
  Done: 1,004,792 rows engineered

  Loading file 5...
  Engineering features...
    File 5: 1,030,516 packets, 16 nodes
  Done: 1,030,516 rows engineered

  Loading file 6...
  Engineering features...
    File 6: 1,020,944 packets, 1